# Tutorial: MLflow for Experiment Tracking and Model Registry

This notebook shows how to use MLflow to track experiments for:
- a simple scikit-learn model
- a PyTorch model
- model registry operations (versions, aliases, loading)

Both model workflows use the census dataset prepared by `process_census_data.py`.

In [ ]:
import gdown
from pathlib import Path
import os

# Install gdown if not already present
# !pip install gdown

# Define the Google Drive file ID from the provided URL
file_id = "1--l9cVrHUtivNwktbvGXyntzAlXfS6_s"

# Define the target directory and filename for the compressed data
# The notebook's DATA_PATH (defined in cell jXS26JMMJfpn) expects
# the uncompressed file at '../data/census-income.data'.
# The _unzip_data function expects the compressed file at DATA_PATH + ".bz2".
# So, the compressed file should be saved as '../data/census-income.data.bz2'.
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True) # Ensure the directory exists

output_filepath = data_dir / "census-income.data"

print(f"Downloading data from Google Drive (ID: {file_id}) to {output_filepath}...")

# Check if the file already exists and is not empty to avoid re-downloading
if output_filepath.exists() and output_filepath.stat().st_size > 0:
    print(f"File '{output_filepath}' already exists. Skipping download.")
else:
    try:
        # Download the file using gdown
        gdown.download(id=file_id, output=str(output_filepath), quiet=False)
        print("Download complete.")
    except Exception as e:
        print(f"Error downloading file: {e}")
        print("Please ensure 'gdown' is installed (`!pip install gdown`) or check the file ID and permissions.")
from pathlib import Path
import bz2
import shutil

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = "../data/census-income.data"

DATA_TYPES = [
    ("age", "float64"),
    ("class of worker", "category"),
    ("detailed industry recode", "category"),
    ("detailed occupation recode", "category"),
    ("education", "category"),
    ("wage per hour", "float64"),
    ("enroll in edu inst last wk", "category"),
    ("marital stat", "category"),
    ("major industry code", "category"),
    ("major occupation code", "category"),
    ("race", "category"),
    ("hispanic origin", "category"),
    ("sex", "category"),
    ("member of a labor union", "category"),
    ("reason for unemployment", "category"),
    ("full or part time employment stat", "category"),
    ("capital gains", "float64"),
    ("capital losses", "float64"),
    ("dividends from stocks", "float64"),
    ("tax filer stat", "category"),
    ("region of previous residence", "category"),
    ("state of previous residence", "category"),
    ("detailed household and family stat", "category"),
    ("detailed household summary in household", "category"),
    ("instance weight_ignore", "float64"),
    ("migration code-change in msa", "category"),
    ("migration code-change in reg", "category"),
    ("migration code-move within reg", "category"),
    ("live in this house 1 year ago", "category"),
    ("migration prev res in sunbelt", "category"),
    ("num persons worked for employer", "float64"),
    ("family members under 18", "category"),
    ("country of birth father", "category"),
    ("country of birth mother", "category"),
    ("country of birth self", "category"),
    ("citizenship", "category"),
    ("own business or self employed", "category"),
    ("fill inc questionnaire for veteran's admin", "category"),
    ("veterans benefits", "category"),
    ("weeks worked in year", "float64"),
    ("year", "category"),
    ("targets", "category"),
]

EDU_CODE = {
    "Children": 0,
    "Less than 1st grade": 1,
    "1st 2nd 3rd or 4th grade": 2,
    "5th or 6th grade": 3,
    "7th and 8th grade": 4,
    "9th grade": 5,
    "10th grade": 6,
    "11th grade": 7,
    "12th grade no diploma": 8,
    "High school graduate": 9,
    "Some college but no degree": 10,
    "Associates degree-academic program": 11,
    "Associates degree-occup /vocational": 12,
    "Bachelors degree(BA AB BS)": 13,
    "Masters degree(MA MS MEng MEd MSW MBA)": 14,
    "Prof school degree (MD DDS DVM LLB JD)": 15,
    "Doctorate degree(PhD EdD)": 15,
}

BINARY_COLUMNS = [
    "member of a labor union",
    "live in this house 1 year ago",
    "own business or self employed",
    "fill inc questionnaire for veteran's admin",
    "veterans benefits",
    "year",
    "sex",
]

def load_raw_data(path):
    """Load the raw census income dataset into a pandas DataFrame.

    Parameters
    ----------
    path : str or pathlib.Path or None
        Path to the uncompressed dataset file. When ``None``, the default
        module path is used and the compressed file is extracted if needed.

    Returns
    -------
    pandas.DataFrame
        Raw census data with column names and dtypes defined by ``DATA_TYPES``.
    """
    if path is None:
        path = DATA_PATH
        if not Path(path).exists():
            _unzip_data()

    return pd.read_csv(path, names=[d[0] for d in DATA_TYPES], dtype=dict(DATA_TYPES))

def _unzip_data():
    """Extract the bundled compressed census income dataset.

    Returns
    -------
    None
        The function writes the extracted file next to the compressed archive.
    """

    source_path = Path(DATA_PATH + ".bz2")
    target_path = source_path.with_suffix("")

    with bz2.open(source_path, "rb") as src, target_path.open("wb") as dst:
        shutil.copyfileobj(src, dst)


def add_education_num(raw_data):
    """Add a numeric encoding of the education label column.

    Parameters
    ----------
    raw_data : pandas.DataFrame
        Input census dataset containing the ``education`` column.

    Returns
    -------
    pandas.DataFrame
        Copy of ``raw_data`` with an additional ``education-num`` column.
    """
    raw_data = raw_data.copy()
    raw_data["education-num"] = np.array(
        [EDU_CODE[v.strip()] for v in raw_data["education"]]
    ).astype("float64")
    return raw_data


def build_features(raw_data):
    """Transform raw census records into a model-ready feature matrix.

    Parameters
    ----------
    raw_data : pandas.DataFrame
        Census records without the target column.

    Returns
    -------
    pandas.DataFrame
        Feature matrix containing one-hot encoded categorical features,
        numerical columns, and factorized binary columns.
    """
    raw_data = add_education_num(raw_data)
    data = raw_data.drop(
        columns=[
            "instance weight_ignore",
            "detailed industry recode",
            "detailed occupation recode",
            "major industry code",
            "major occupation code",
            "country of birth father",
            "country of birth mother",
            "country of birth self",
            "state of previous residence",
            "detailed household and family stat",
            "education",
        ]
    )

    binary_data = data[BINARY_COLUMNS].copy()
    categorical_data = data.select_dtypes(include=["category"]).drop(
        columns=BINARY_COLUMNS, errors="ignore"
    )
    numerical_data = data.select_dtypes(include=["float64", "int64"]).drop(
        columns=BINARY_COLUMNS, errors="ignore"
    )

    binary_data[
        (binary_data == " 2")
        | (binary_data == " ?")
        | (binary_data == " Not in universe")
        | (binary_data == " Not in universe under 1 year old")
    ] = np.nan
    binary_data = binary_data.apply(lambda x: pd.factorize(x)[0])

    encoder = OneHotEncoder(handle_unknown="ignore")
    encoder.fit(categorical_data)

    categorical_source = data.select_dtypes(include=["category"]).drop(
        columns=BINARY_COLUMNS, errors="ignore"
    )

    return pd.concat(
        [
            pd.DataFrame(
                encoder.transform(categorical_source).toarray(),
                index=categorical_source.index,
                columns=encoder.get_feature_names_out(categorical_source.columns),
            ),
            numerical_data,
            binary_data,
        ],
        axis=1,
    )


def rebalance_training_data(train_x, train_y, random_state=99):
    """Downsample the majority class to create a balanced training set.

    Parameters
    ----------
    train_x : pandas.DataFrame
        Training features.
    train_y : pandas.Series
        Binary target labels aligned with ``train_x``.
    random_state : int, default=99
        Seed used for deterministic resampling.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.Series]
        Resampled training features and labels with balanced classes.
    """
    rng = np.random.default_rng(random_state)
    positive_idx = train_y.index.values[train_y == 1]
    negative_idx = rng.permutation(train_y.index.values[train_y == 0])
    index = rng.permutation(np.hstack((positive_idx, negative_idx))[: 2 * len(positive_idx)])
    return train_x.loc[index], train_y.loc[index]


def prepare_census_data(
    path=None,
    random_state=99,
    split_validation=False,
    rebalance=True,
):
    """Load, featurize, split, and optionally rebalance the census dataset.

    Parameters
    ----------
    path : str or pathlib.Path or None, default=None
        Path to the raw census data file. When ``None``, the default dataset
        path is used.
    random_state : int, default=99
        Seed used for train-test splitting and optional rebalancing.
    split_validation : bool, default=False
        Whether to create a separate validation split in addition to the test
        split.
    rebalance : bool, default=True
        Whether to rebalance the training data after splitting.

    Returns
    -------
    tuple
        Either ``(train_x, test_x, train_y, test_y)`` or
        ``(train_x, val_x, test_x, train_y, val_y, test_y)`` depending on
        ``split_validation``.
    """
    raw_data = load_raw_data(path)
    targets, _ = pd.factorize(raw_data["targets"])
    features = build_features(raw_data.drop(columns=["targets"]))
    target_series = pd.Series(targets, index=features.index)

    if split_validation:
        train_x, temp_x, train_y, temp_y = train_test_split(
            features, target_series, random_state=random_state
        )
        val_x, test_x, val_y, test_y = train_test_split(
            temp_x, temp_y, random_state=random_state
        )
        if rebalance:
            train_x, train_y = rebalance_training_data(train_x, train_y, random_state)
        return train_x, val_x, test_x, train_y, val_y, test_y

    train_x, test_x, train_y, test_y = train_test_split(
        features, target_series, random_state=random_state
    )
    if rebalance:
        train_x, train_y = rebalance_training_data(train_x, train_y, random_state)
    return train_x, test_x, train_y, test_y

In [ ]:
!pip install mlflow skops

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import mlflow
import mlflow.pytorch
import mlflow.sklearn
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from mlflow.tracking import MlflowClient
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

Configure MLflow tracking

In [ ]:
MLFLOW_DIR = PROJECT_ROOT / "mlruns_tutorial"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_URI = MLFLOW_DIR.as_uri()

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

In [ ]:
EXPERIMENT_NAME = "mlflow_tutorial_experiment"
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
client = MlflowClient()

Track a simple scikit-learn model

In [ ]:
X_train, X_test, y_train, y_test = prepare_census_data(
    path=str(PROJECT_ROOT / "data" / "census-income.data"),
    random_state=42,
    rebalance=False,
    split_validation=False,
 )

In [ ]:
X_train.shape, X_test.shape

In [ ]:
sklearn_configs = [
    {"C": 0.1, "max_iter": 300},
    {"C": 1.0, "max_iter": 500},
    {"C": 5.0, "max_iter": 700},
]

In [ ]:
sklearn_run_ids = []
sklearn_results = []

for cfg in sklearn_configs:
    simple_model = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=cfg["max_iter"],
                    C=cfg["C"],
                    random_state=42,
                ),
            ),
        ]
    )

    with mlflow.start_run(run_name=f"sklearn_logreg_C_{cfg['C']}") as run:
        simple_model.fit(X_train, y_train)
        preds = simple_model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        mlflow.log_param("model_type", "logistic_regression")
        mlflow.log_param("C", cfg["C"] )
        mlflow.log_param("max_iter", cfg["max_iter"] )
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1", f1)
        mlflow.sklearn.log_model(
            simple_model,
            name="simple_model",
            serialization_format=SKLEARN_SERIALIZATION_FORMAT,
        )

        sklearn_run_ids.append(run.info.run_id)
        sklearn_results.append({"run_id": run.info.run_id, "C": cfg["C"], "max_iter": cfg["max_iter"], "accuracy": acc, "f1": f1})

In [ ]:
sklearn_results_df = pd.DataFrame(sklearn_results).sort_values("f1", ascending=False).reset_index(drop=True)
best_sklearn_run_id = sklearn_results_df.loc[0, "run_id"]
sklearn_run_id = best_sklearn_run_id
sklearn_results_df

Track a PyTorch model

In [ ]:
torch.manual_seed(42)

In [ ]:
X_train_t = torch.from_numpy(X_train.to_numpy(dtype="float32"))

In [ ]:
y_train_t = torch.from_numpy(y_train.to_numpy(dtype="float32")).view(-1, 1)

In [ ]:
X_test_t = torch.from_numpy(X_test.to_numpy(dtype="float32"))

In [ ]:
y_test_t = torch.from_numpy(y_test.to_numpy(dtype="float32")).view(-1, 1)

In [ ]:
torch_configs = [
    {"hidden_units": 8, "lr": 0.01, "epochs": 30},
    {"hidden_units": 16, "lr": 0.01, "epochs": 40},
    {"hidden_units": 32, "lr": 0.005, "epochs": 50},
]

In [ ]:
torch_run_ids = []
torch_results = []

In [ ]:
len(torch_configs)

In [ ]:
torch_configs

In [ ]:
for cfg in torch_configs:
    torch.manual_seed(42)
    torch_model = nn.Sequential(
        nn.Linear(X_train.shape[1], cfg["hidden_units"]),
        nn.ReLU(),
        nn.Linear(cfg["hidden_units"], 1),
        nn.Sigmoid(),
    )
    criterion = nn.BCELoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=cfg["lr"] )

    with mlflow.start_run(run_name=f"torch_mlp_h{cfg['hidden_units']}_lr{cfg['lr']}") as run:
        mlflow.log_param("model_type", "torch_mlp")
        mlflow.log_param("hidden_units", cfg["hidden_units"] )
        mlflow.log_param("lr", cfg["lr"] )
        mlflow.log_param("epochs", cfg["epochs"] )

        for epoch in range(cfg["epochs"]):
            torch_model.train()
            optimizer.zero_grad()
            outputs = torch_model(X_train_t)
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 10 == 0:
                mlflow.log_metric("train_loss", float(loss.item()), step=epoch + 1)

        torch_model.eval()
        with torch.no_grad():
            test_probs = torch_model(X_test_t)
            test_preds = (test_probs >= 0.5).float()
            test_acc = (test_preds.eq(y_test_t)).float().mean().item()

        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.pytorch.log_model(torch_model, name="torch_model")

        torch_run_ids.append(run.info.run_id)
        torch_results.append({
            "run_id": run.info.run_id,
            "hidden_units": cfg["hidden_units"],
            "lr": cfg["lr"],
            "epochs": cfg["epochs"],
            "test_accuracy": test_acc,
        })

In [ ]:
torch_results_df = pd.DataFrame(torch_results).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
best_torch_run_id = torch_results_df.loc[0, "run_id"]
torch_run_id = best_torch_run_id
torch_results_df

Compare runs

In [ ]:
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

In [ ]:
cols = [
    "run_id",
    "tags.mlflow.runName",
    "params.model_type",
    "params.C",
    "params.max_iter",
    "params.hidden_units",
    "params.lr",
    "params.epochs",
    "metrics.accuracy",
    "metrics.f1",
    "metrics.test_accuracy",
]

In [ ]:
runs_df[cols].fillna("-")

Register models in Model Registry

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

simple_registered_name = f"tutorial_simple_model_{timestamp}"
torch_registered_name = f"tutorial_torch_model_{timestamp}"

simple_uri = f"runs:/{sklearn_run_id}/simple_model"
torch_uri = f"runs:/{torch_run_id}/torch_model"

simple_version = mlflow.register_model(model_uri=simple_uri, name=simple_registered_name)
torch_version = mlflow.register_model(model_uri=torch_uri, name=torch_registered_name)

In [ ]:
simple_version.version, torch_version.version

Add aliases

In [ ]:
client.set_registered_model_alias(simple_registered_name, "best", simple_version.version)

In [ ]:
client.set_registered_model_alias(torch_registered_name, "candidate", torch_version.version)

Load models from registry

In [ ]:
loaded_simple = mlflow.sklearn.load_model(f"models:/{simple_registered_name}@champion")

In [ ]:
loaded_torch = mlflow.pytorch.load_model(f"models:/{torch_registered_name}@candidate")

In [ ]:
loaded_simple.predict(X_test[:5])

In [ ]:
loaded_torch.eval()

In [ ]:
with torch.no_grad():
    loaded_probs = loaded_torch(X_test_t[:5])
loaded_probs

Inspect registry entries

In [ ]:
client.get_registered_model(simple_registered_name).aliases

In [ ]:
client.get_registered_model(torch_registered_name).aliases

Open the MLflow UI

In [ ]:
mlflow_ui_cmd = [
    "mlflow",
    "ui",
    "--backend-store-uri",
    TRACKING_URI,
    "--host",
    "127.0.0.1",
    "--port",
    "5000",
]

mlflow_ui_process = subprocess.Popen(
    mlflow_ui_cmd,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS,
)

print(f"MLflow UI started at http://127.0.0.1:5000 using store {TRACKING_URI}")
mlflow_ui_process.pid

In [ ]:
MLFLOW_DIR

Run the previous code cell to start the MLflow UI server in the background.

If running from a terminal, use a URI (not a raw Windows path):

`mlflow ui --backend-store-uri file:///C:/Users/NC3769/Code/AI_methods_in_economics_and_finance_research/mlruns_tutorial`

Then open http://127.0.0.1:5000 to browse experiments and registry.